# Do ajuste de curvas ao aprendizado de máquina

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/09-aprendizado-de-maquina.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. ver uma rede neural como um modelo de mínimos quadrados não linear, com muitos parâmetros;
> 2. entender o treino como um problema de otimização resolvido por gradiente (capítulo 5), com derivadas calculadas por diferenciação automática (capítulo 10);
> 3. preparar os dados (escalas, logaritmos) para que esse problema seja bem condicionado;
> 4. separar dados de treino e de validação e reconhecer o sobreajuste;
> 5. montar uma rede informada pela física (PINN) e saber quando ela é, ou não é, a ferramenta certa.

Este capítulo usa **PyTorch**, que não roda no navegador. Para executar as células, use o Colab (o PyTorch já vem instalado) ou uma instalação local. Os treinos são pequenos e levam segundos num processador comum.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

In [ ]:
import torch
from torch import nn
from cnlivro.redes import mlp, treinar, Padronizador

torch.set_default_dtype(torch.float64)     # 16 algarismos: facilita comparar com os métodos clássicos

## 1. Problema de engenharia: o arrasto de uma esfera

Quanto uma esfera é freada por um fluido? A força de arrasto é

$$
F_D = C_D\,\frac{1}{2}\rho v^2 A,
$$

e toda a física difícil está no **coeficiente de arrasto** $C_D$, que depende do número de Reynolds $\text{Re} = \rho v d/\mu$. Para grãos de areia num decantador, $\text{Re}$ é da ordem de 1; para uma bola de futebol chutada, da ordem de $10^5$. Um mesmo simulador precisa de $C_D$ numa faixa de **sete ordens de grandeza**.

Não há fórmula teórica para toda a faixa. Para $\text{Re} \ll 1$ vale a lei de Stokes, $C_D = 24/\text{Re}$. Entre $10^3$ e $2 \times 10^5$, $C_D$ fica perto de $0{,}4$ a $0{,}5$. Perto de $\text{Re} = 3 \times 10^5$, a camada limite fica turbulenta, se descola mais tarde, e $C_D$ despenca para cerca de $0{,}1$: é a **crise do arrasto**, que explica as covinhas das bolas de golfe.

Temos 60 medidas de túnel de vento e de tanque. Queremos uma função $C_D(\text{Re})$ que passe pelo meio dos dados, sem saber de antemão a sua forma. Os dados abaixo são **sintéticos**: gerados com uma correlação empírica publicada (Morrison, 2013) e um ruído de 3 %, para que conheçamos a curva verdadeira.

In [ ]:
def cd_correlacao(Re):
    # correlação empírica de F. A. Morrison (2013) para esferas lisas, 0,1 < Re < 10⁶
    Re = np.asarray(Re, dtype=float)
    return (24 / Re + 2.6 * (Re / 5) / (1 + (Re / 5) ** 1.52)
            + 0.411 * (Re / 2.63e5) ** -7.94 / (1 + (Re / 2.63e5) ** -8.0)
            + 0.25 * (Re / 1e6) / (1 + Re / 1e6))

rng = np.random.default_rng(3)
Re_med = np.logspace(-1, 6, 60) * np.exp(rng.normal(0, 0.05, 60))
Cd_med = cd_correlacao(Re_med) * np.exp(rng.normal(0, 0.03, 60))       # ruído de 3 %
print(f"Re de {Re_med.min():.2f} a {Re_med.max():.2e};  C_D de {Cd_med.min():.3f} a {Cd_med.max():.0f}")

In [ ]:
Re_curva = np.logspace(-1, 6, 400)
fig = go.Figure()
fig.add_trace(go.Scatter(x=Re_curva, y=24 / Re_curva, mode="lines", name="Stokes: 24/Re",
                         line=dict(color=CORES["referencia"], dash="dash")))
fig.add_trace(go.Scatter(x=Re_curva, y=cd_correlacao(Re_curva), mode="lines", name="curva verdadeira (correlação)",
                         line=dict(color=CORES["exata"])))
fig.add_trace(go.Scatter(x=Re_med, y=Cd_med, mode="markers", name="medidas", marker=dict(color=CORES["aproximacao"], size=7)))
fig.add_annotation(x=np.log10(3e5), y=np.log10(0.09), text="crise do arrasto", showarrow=True, ax=-130, ay=15)
fig.update_xaxes(type="log", title="número de Reynolds", exponentformat="power")
fig.update_yaxes(type="log", title="coeficiente de arrasto C_D", range=[-1.2, 2.6])
fig.update_layout(width=800, height=460, legend=dict(x=0.55, y=0.98))
mostrar(fig)

**O que notar:** em escala log-log, os dados formam uma curva suave com três regimes e um degrau. Um polinômio em $\log\text{Re}$ teria dificuldade com o degrau; uma soma de termos como a da correlação exige conhecer a física. Uma rede neural é uma forma de ajustar uma curva **sem escolher a forma** antes.

## 2. A ideia em uma imagem

Uma rede neural com uma camada oculta é uma soma de "degraus suaves":

$$
N(x) = c_0 + \sum_{j=1}^{m} c_j \tanh\big(w_j x + b_j\big).
$$

Cada neurônio $\tanh(w_j x + b_j)$ é um degrau que sobe perto de $x = -b_j/w_j$, com inclinação controlada por $w_j$. Somando degraus com alturas $c_j$, dá para montar quase qualquer curva. Na figura, os degraus estão fixos (posições igualmente espaçadas) e só as alturas $c_j$ são ajustadas, por **mínimos quadrados lineares**, exatamente como no capítulo 7. Arraste o slider para usar mais neurônios.

In [ ]:
x_ilus = np.linspace(-1, 1, 41)
y_ilus = np.exp(-3 * x_ilus**2) * np.cos(4 * x_ilus) + 0.3 * x_ilus
xx = np.linspace(-1, 1, 400)

def quadro_neuronios(m):
    centros = np.linspace(-1, 1, m + 2)[1:-1] if m > 1 else np.array([0.0])
    w = 2.5 * (m + 1)
    base = lambda x: np.column_stack([np.ones_like(x)] + [np.tanh(w * (x - c_)) for c_ in centros])
    coef, *_ = np.linalg.lstsq(base(x_ilus), y_ilus, rcond=None)
    fig = go.Figure()
    for j, c_ in enumerate(centros):
        fig.add_trace(go.Scatter(x=xx, y=coef[j + 1] * np.tanh(w * (xx - c_)), mode="lines",
                                 line=dict(color=CORES["referencia"], width=1), opacity=0.6,
                                 name="neurônios c_j tanh(w(x − x_j))", showlegend=(j == 0)))
    fig.add_trace(go.Scatter(x=x_ilus, y=y_ilus, mode="markers", name="dados", marker=dict(color=CORES["exata"], size=6)))
    fig.add_trace(go.Scatter(x=xx, y=base(xx) @ coef, mode="lines", name="soma = rede", line=dict(color=CORES["aproximacao"], width=3)))
    rms = np.sqrt(np.mean((base(x_ilus) @ coef - y_ilus) ** 2))
    fig.update_xaxes(title="x")
    fig.update_yaxes(title="y", range=[-1.3, 1.5])
    fig.update_layout(width=780, height=440, legend=dict(orientation="h", y=-0.2),
                      title=f"{m} neurônio{'s' if m > 1 else ''}: erro RMS = {numero(rms, 3)}")
    return fig

quadros = [(str(m), quadro_neuronios(m)) for m in [1, 2, 4, 8, 16]]
mostrar(com_controle(quadros, rotulo="neurônios: ", ativo=2, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** com um neurônio, só um degrau. Com 8 ou 16, a soma acompanha os dados. Esse é o **teorema da aproximação universal**: com neurônios suficientes, uma rede de uma camada aproxima qualquer função contínua num intervalo, com o erro que se quiser. O teorema não diz quantos neurônios são precisos, nem como achar os pesos.

Numa rede de verdade, as posições e inclinações $w_j, b_j$ **também** são ajustadas. Aí o problema deixa de ser linear nos parâmetros, e os mínimos quadrados viram um problema de **otimização não linear**.

## 3. Formulação matemática

### 3.1 Uma rede é um modelo com muitos parâmetros

Uma rede totalmente conectada (*multilayer perceptron*, MLP) com $L$ camadas ocultas compõe transformações afins com uma função de ativação $\sigma$ aplicada elemento a elemento:

$$
\mathbf{h}_0 = \mathbf{x}, \qquad
\mathbf{h}_\ell = \sigma\left(W_\ell\,\mathbf{h}_{\ell-1} + \mathbf{b}_\ell\right),\ \ell = 1, \ldots, L, \qquad
N(\mathbf{x}; \boldsymbol\theta) = W_{L+1}\,\mathbf{h}_L + \mathbf{b}_{L+1}.
$$

O vetor $\boldsymbol\theta$ reúne todas as matrizes $W_\ell$ e vetores $\mathbf{b}_\ell$. Uma rede com 3 camadas de 32 neurônios e uma entrada já tem $64 + 2 \times 1056 + 33 = 2209$ parâmetros. As ativações mais usadas são $\tanh$ e a ReLU, $\max(0, z)$; para ajustar curvas suaves e para as PINNs da seção 3.5, $\tanh$ é a mais indicada, porque é suave.

### 3.2 Treinar é resolver mínimos quadrados não lineares

Com dados $(\mathbf{x}_i, y_i)$, $i = 1, \ldots, n$, o treino minimiza o **erro quadrático médio**:

$$
\mathcal{L}(\boldsymbol\theta) = \frac{1}{n}\sum_{i=1}^{n}\big(N(\mathbf{x}_i; \boldsymbol\theta) - y_i\big)^2 .
$$ (eq-perda)

É o mesmo problema do capítulo 7, com um modelo não linear nos parâmetros. Com milhares de parâmetros, o método de Gauss-Newton seria caro, e usa-se **descida pelo gradiente** (capítulo 5):

$$
\boldsymbol\theta_{k+1} = \boldsymbol\theta_k - \eta\,\nabla\mathcal{L}(\boldsymbol\theta_k).
$$

O passo $\eta$ se chama **taxa de aprendizado**. O gradiente é calculado por **diferenciação automática no modo reverso** (capítulo 10), que dá as 2209 derivadas parciais por um custo de poucas avaliações da rede. Na prática se usa o **Adam**, uma descida pelo gradiente que guarda médias móveis do gradiente e do seu quadrado para ajustar o passo de cada parâmetro. Quando os dados são muitos, cada passo usa só um sorteio deles (gradiente **estocástico**, em minilotes).

Três diferenças em relação ao capítulo 5: a função $\mathcal{L}$ **não é convexa** e tem muitos mínimos locais; não se procura o mínimo exato (seção 3.4); e o resultado depende do **ponto de partida** aleatório.

### 3.3 Escalas: condicionamento do treino

Se uma entrada varia de $0{,}1$ a $10^6$, os neurônios da primeira camada recebem $w_j x + b_j$ com valores enormes, e $\tanh$ **satura**: vale $\pm 1$ para quase todos os dados, com derivada praticamente zero. O gradiente some e o treino para. Mesmo sem saturação, variáveis com escalas muito diferentes tornam a hessiana de $\mathcal{L}$ mal condicionada, e a descida pelo gradiente anda em zigue-zague (capítulo 5).

A receita é **padronizar**: transformar cada entrada e cada saída para média 0 e desvio padrão 1. Quando uma grandeza varia em ordens de grandeza, o certo é padronizar o seu **logaritmo**. Ajustar $\log C_D$ em vez de $C_D$ tem outra vantagem: o erro quadrático em $\log C_D$ mede o erro **relativo**, que é o que importa quando $C_D$ vai de $0{,}1$ a 200.

### 3.4 Sobreajuste e validação

Com parâmetros suficientes, uma rede passa por **todos** os pontos de treino, inclusive pelo ruído. É o mesmo fenômeno do polinômio de grau alto dos capítulos 6 e 7. A defesa é separar uma parte dos dados para **validação**, que o treino não vê:

- se a perda de treino cai e a de validação também, o modelo está aprendendo;
- se a perda de treino continua caindo e a de validação **sobe**, o modelo está decorando o ruído.

Um sinal simples: se a perda de treino fica **abaixo da variância do ruído** das medidas, a rede está ajustando o ruído. As defesas são parar o treino no mínimo da validação (*early stopping*), usar menos parâmetros, ou penalizar pesos grandes ($\mathcal{L} + \lambda\|\boldsymbol\theta\|^2$), que é a regularização de Tikhonov do capítulo 7.

### 3.5 Redes informadas pela física (PINNs)

E se, em vez de dados, tivermos uma **equação diferencial**? Numa aleta de resfriamento de comprimento $L$, com $\theta = (T - T_\infty)/(T_b - T_\infty)$ e $\xi = x/L$,

$$
\frac{d^2\theta}{d\xi^2} - (mL)^2\,\theta = 0, \qquad \theta(0) = 1, \qquad \theta'(1) = 0,
$$ (eq-aleta)

com $m^2 = hP/(kA_c)$. A ideia de uma PINN (*physics-informed neural network*) é usar uma rede $\theta_N(\xi)$ como solução candidata e colocar a **equação** na perda, num conjunto de pontos de colocação $\xi_i$:

$$
\mathcal{L} = \frac{1}{n}\sum_i \Big(\theta_N''(\xi_i) - (mL)^2\theta_N(\xi_i)\Big)^2 + \big(\theta_N(0) - 1\big)^2 + \theta_N'(1)^2 .
$$

As derivadas $\theta_N'$ e $\theta_N''$ da rede em relação à **entrada** são calculadas pela própria diferenciação automática. Nenhum dado é necessário, e dados medidos, quando existem, entram como mais um termo da perda. O mesmo problema tem solução exata, $\theta = \cosh(mL(1 - \xi))/\cosh(mL)$, e é resolvido em milissegundos por diferenças finitas (capítulo 14): bom para conferir.

## 4. À mão

Um neurônio, um dado: $N(x) = c\tanh(wx + b)$, com o ponto $(x, y) = (1;\ 0{,}5)$, partindo de $w = 0{,}5$, $b = 0$, $c = 1$. A perda é $\mathcal{L} = (N - y)^2$.

**Passada para a frente** (*forward*):

$$
z = wx + b = 0{,}5, \qquad a = \tanh z = 0{,}462117, \qquad N = ca = 0{,}462117,
$$
$$
r = N - y = -0{,}037883, \qquad \mathcal{L} = r^2 = 0{,}0014351.
$$

**Passada para trás** (*backward*), pela regra da cadeia, da saída para a entrada:

$$
\frac{\partial \mathcal{L}}{\partial c} = 2r\,a = -0{,}035013, \qquad
\frac{\partial \mathcal{L}}{\partial z} = 2r\,c\,(1 - a^2) = -0{,}059586,
$$
$$
\frac{\partial \mathcal{L}}{\partial w} = \frac{\partial \mathcal{L}}{\partial z}\,x = -0{,}059586, \qquad
\frac{\partial \mathcal{L}}{\partial b} = \frac{\partial \mathcal{L}}{\partial z} = -0{,}059586 .
$$

Repare que $\partial\mathcal{L}/\partial z$ foi calculado **uma vez** e reaproveitado para $w$ e $b$: é a economia do modo reverso.

**Um passo de descida** com $\eta = 0{,}5$: $c = 1{,}017507$, $w = 0{,}529793$, $b = 0{,}029793$. Agora $z = 0{,}559586$, $N = 1{,}017507 \times \tanh(0{,}559587) = 0{,}516547$, e a perda cai para $2{,}7 \times 10^{-4}$, cinco vezes menor. Com $\eta = 1$, o passo passaria do ponto: $N$ iria a $0{,}57$ e a perda **subiria** para $4{,}9 \times 10^{-3}$. A taxa de aprendizado é o passo da descida pelo gradiente, com os mesmos riscos do capítulo 5.

In [ ]:
w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
c = torch.tensor(1.0, requires_grad=True)
x, y = 1.0, 0.5
perda = (c * torch.tanh(w * x + b) - y) ** 2
perda.backward()
print(f"perda = {perda.item():.7f}")
print(f"dL/dc = {c.grad.item():.6f},  dL/dw = {w.grad.item():.6f},  dL/db = {b.grad.item():.6f}")
for eta in [0.5, 1.0]:
    with torch.no_grad():
        c2, w2, b2 = c - eta * c.grad, w - eta * w.grad, b - eta * b.grad
        print(f"η = {eta}: nova perda = {((c2 * torch.tanh(w2 * x + b2) - y) ** 2).item():.2e}")

## 5. Explore

### 5.1 Sobreajuste ao vivo

Uma bomba centrífuga foi ensaiada em 14 vazões; a eficiência foi medida com incerteza de 0,02 (dois pontos percentuais). Outras 40 medidas, de um segundo ensaio, ficam guardadas para validação. Treinamos uma rede grande (3 camadas de 64 neurônios, 8 513 parâmetros, para 14 pontos!) e guardamos a curva em vários momentos do treino. Arraste o slider para avançar nas épocas.

In [ ]:
eficiencia = lambda q: 0.85 * q * (2.2 - q) * np.exp(-0.15 * q)       # curva verdadeira (desconhecida)
rng = np.random.default_rng(2)
q_tr = np.sort(rng.uniform(0.1, 1.9, 14))
e_tr = eficiencia(q_tr) + rng.normal(0, 0.02, 14)
q_va = np.sort(rng.uniform(0.1, 1.9, 40))
e_va = eficiencia(q_va) + rng.normal(0, 0.02, 40)
escala = lambda q: ((np.asarray(q) - 1.0) / 0.55)[:, None]           # padroniza a vazão (média 1, desvio ≈ 0,55)
qq = np.linspace(0.05, 1.95, 300)

torch.manual_seed(1)
instantes = [50, 300, 700, 2000, 8000]
r_bomba = treinar(mlp(largura=64, camadas=3), escala(q_tr), e_tr[:, None], epocas=8000, lr=3e-3,
                  x_val=escala(q_va), y_val=e_va[:, None], instantes=instantes,
                  callback=lambda m: m(torch.as_tensor(escala(qq))).numpy().ravel())
h_tr, h_va = np.array(r_bomba.historico), np.array(r_bomba.historico_validacao)
melhor = int(np.argmin(h_va)) + 1
print(f"menor perda de validação: {h_va.min():.1e} na época {melhor};  no fim: treino {h_tr[-1]:.1e}, validação {h_va[-1]:.1e}")
print(f"variância do ruído das medidas: {0.02**2:.1e}")

def quadro_epoca(ep):
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1, column_widths=[0.55, 0.45])
    fig.add_trace(go.Scatter(x=qq, y=eficiencia(qq), mode="lines", name="curva verdadeira",
                             line=dict(color=CORES["exata"], dash="dot")), 1, 1)
    fig.add_trace(go.Scatter(x=q_va, y=e_va, mode="markers", name="validação",
                             marker=dict(color=CORES["referencia"], size=6)), 1, 1)
    fig.add_trace(go.Scatter(x=q_tr, y=e_tr, mode="markers", name="treino",
                             marker=dict(color=CORES["exata"], size=9)), 1, 1)
    fig.add_trace(go.Scatter(x=qq, y=r_bomba.instantaneos[ep], mode="lines", name="rede",
                             line=dict(color=CORES["aproximacao"], width=3)), 1, 1)
    epocas = np.arange(1, h_tr.size + 1)
    fig.add_trace(go.Scatter(x=epocas, y=h_tr, mode="lines", name="perda de treino",
                             line=dict(color=CORES["aproximacao"])), 1, 2)
    fig.add_trace(go.Scatter(x=epocas, y=h_va, mode="lines", name="perda de validação",
                             line=dict(color=CORES["erro"])), 1, 2)
    fig.add_trace(go.Scatter(x=[ep, ep], y=[1e-5, 1e-1], mode="lines", name="época atual",
                             line=dict(color=CORES["destaque"], dash="dash")), 1, 2)
    fig.update_xaxes(title="vazão (normalizada)", row=1, col=1)
    fig.update_yaxes(title="eficiência", range=[0, 1.1], row=1, col=1)
    fig.update_xaxes(title="época", type="log", row=1, col=2)
    fig.update_yaxes(title="perda (EQM)", type="log", exponentformat="power", row=1, col=2)
    fig.update_layout(width=980, height=440, legend=dict(orientation="h", y=-0.22), title=f"época {ep}")
    return fig

quadros = [(str(ep), quadro_epoca(ep)) for ep in instantes]
mostrar(com_controle(quadros, rotulo="época ", ativo=2, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** no começo, a rede é rígida demais e não acompanha os dados. Por volta da época 700, a perda de validação passa pelo mínimo, e a curva está próxima da verdadeira. Depois disso, a perda de treino continua caindo, **abaixo da variância do ruído** ($4 \times 10^{-4}$), enquanto a de validação sobe: a rede passa a se torcer para acertar o ruído de cada ponto de treino. Sem os dados de validação, a época 8000 pareceria a melhor.

### 5.2 Uma PINN aprendendo a física da aleta

A rede abaixo nunca vê uma temperatura medida: só a equação {eq}`eq-aleta` com $mL = 1{,}5$ e as condições de contorno. Arraste o slider para acompanhar o treino.

In [ ]:
def treina_pinn(mL, epocas=4000, n_col=64, semente=0, instantes=()):
    torch.manual_seed(semente)
    rede = mlp(largura=24, camadas=3)
    xi = torch.linspace(0, 1, n_col)[:, None].requires_grad_(True)        # pontos de colocação
    xi0 = torch.zeros(1, 1)
    xi1 = torch.ones(1, 1, requires_grad=True)
    otim = torch.optim.Adam(rede.parameters(), lr=5e-3)
    historico, fotos = [], {}
    xx = torch.linspace(0, 1, 201)[:, None]
    for ep in range(1, epocas + 1):
        otim.zero_grad()
        th = rede(xi)
        d1 = torch.autograd.grad(th.sum(), xi, create_graph=True)[0]      # θ'(ξ) pela dif. automática
        d2 = torch.autograd.grad(d1.sum(), xi, create_graph=True)[0]      # θ''(ξ)
        th1 = rede(xi1)
        d1_ponta = torch.autograd.grad(th1.sum(), xi1, create_graph=True)[0]
        perda = ((d2 - mL**2 * th) ** 2).mean() + ((rede(xi0) - 1) ** 2).sum() + (d1_ponta**2).sum()
        perda.backward()
        otim.step()
        historico.append(perda.item())
        if ep in instantes:
            with torch.no_grad():
                fotos[ep] = rede(xx).numpy().ravel()
    with torch.no_grad():
        final = rede(xx).numpy().ravel()
    return final, historico, fotos

xi_plot = np.linspace(0, 1, 201)
exata_aleta = lambda mL: np.cosh(mL * (1 - xi_plot)) / np.cosh(mL)
momentos = [10, 100, 500, 1500, 4000]
theta_pinn, hist_pinn, fotos_pinn = treina_pinn(1.5, instantes=momentos)
print(f"erro máximo da PINN no fim: {np.abs(theta_pinn - exata_aleta(1.5)).max():.1e}")

def quadro_pinn(ep):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=xi_plot, y=exata_aleta(1.5), mode="lines", name="solução exata",
                             line=dict(color=CORES["exata"], dash="dot", width=3)))
    fig.add_trace(go.Scatter(x=xi_plot, y=fotos_pinn[ep], mode="lines", name="PINN",
                             line=dict(color=CORES["aproximacao"], width=3)))
    fig.update_xaxes(title="posição ξ = x/L")
    fig.update_yaxes(title="θ = (T − T∞)/(T_b − T∞)", range=[-0.2, 1.2])
    fig.update_layout(width=720, height=420, legend=dict(x=0.6, y=0.98),
                      title=f"época {ep}: perda = {hist_pinn[ep - 1]:.1e}".replace(".", ","))
    return fig

quadros = [(str(ep), quadro_pinn(ep)) for ep in momentos]
mostrar(com_controle(quadros, rotulo="época ", ativo=1, titulos=[q[1].layout.title.text for q in quadros]))

**O que notar:** no começo, a rede satisfaz mal a condição $\theta(0) = 1$ e a equação. Com o treino, ela se encaixa na solução exata, sem nenhum dado de temperatura. O preço: alguns milhares de passos de otimização para um problema que as diferenças finitas resolvem com um sistema tridiagonal. A força das PINNs aparece em **problemas inversos**, quando parte da física é desconhecida (por exemplo, o coeficiente de convecção $h$) e há algumas medidas para combinar com a equação (exercício 8).

## 6. Trabalhando com o agente

### 6.1 O pedido e o código

> *"Ajuste uma rede neural em PyTorch (3 camadas ocultas de 32 neurônios, tanh, Adam) aos dados de C_D × Re e devolva uma função `cd_rede(Re)`. Mostre a perda final e o R² do ajuste."*

Um código típico:

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
torch.manual_seed(0)
X = torch.tensor(Re_med).reshape(-1, 1)
Y = torch.tensor(Cd_med).reshape(-1, 1)
modelo = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(),
                       nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))
otimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)
for epoca in range(5000):
    otimizador.zero_grad()
    perda = ((modelo(X) - Y) ** 2).mean()
    perda.backward()
    otimizador.step()

def cd_rede(Re):
    with torch.no_grad():
        return modelo(torch.tensor(np.atleast_1d(Re), dtype=torch.float64).reshape(-1, 1)).numpy().ravel()

r2 = 1 - np.sum((cd_rede(Re_med) - Cd_med) ** 2) / np.sum((Cd_med - Cd_med.mean()) ** 2)
print(f"perda final (EQM): {perda.item():.1f}")
print(f"R² = {r2:.4f}")

O treino terminou sem erros, e um $R^2$ de 0,89 parece razoável para dados ruidosos. Uma rede de 2209 parâmetros que explica a maior parte da variância dos dados: o que pode estar errado?

### 6.2 Verificação

**Gráfico na escala certa.** Os dados variam em ordens de grandeza; o gráfico precisa ser log-log:

In [ ]:
Re_curva = np.logspace(-1, 6, 400)
fig = go.Figure()
fig.add_trace(go.Scatter(x=Re_curva, y=cd_correlacao(Re_curva), mode="lines", name="curva verdadeira",
                         line=dict(color=CORES["exata"], dash="dot")))
fig.add_trace(go.Scatter(x=Re_med, y=Cd_med, mode="markers", name="medidas", marker=dict(color=CORES["exata"], size=6)))
fig.add_trace(go.Scatter(x=Re_curva, y=cd_rede(Re_curva), mode="lines", name="rede do agente",
                         line=dict(color=CORES["erro"], width=3)))
fig.update_xaxes(type="log", title="número de Reynolds", exponentformat="power")
fig.update_yaxes(type="log", title="C_D", range=[-1.2, 2.6])
fig.update_layout(width=800, height=440, legend=dict(x=0.6, y=0.98))
mostrar(fig)

A partir de algumas centenas de Reynolds, a rede devolve uma **constante** e não vê nem o resto da curva nem a crise do arrasto. No outro extremo, abaixo de $\text{Re} \approx 0{,}2$, ela também achata e se afasta da lei de Stokes.

**Limites físicos e erro relativo.** Duas verificações que não dependem de gráfico:

In [ ]:
for Re_ in [0.1, 0.5]:
    print(f"Re = {Re_}: rede {cd_rede(Re_)[0]:6.1f};  Stokes 24/Re = {24 / Re_:6.1f}")
erro_rel = np.abs(cd_rede(Re_med) / Cd_med - 1)
print(f"erro relativo nos dados: mediana {np.median(erro_rel):.0%}, máximo {erro_rel.max():.0%}")
print(f"C_D em Re = 3×10⁵ (crise): rede {cd_rede(3e5)[0]:.2f}, curva verdadeira {cd_correlacao(3e5):.2f}")

O $R^2$ enganou porque é calculado com o erro **absoluto**: a variância dos dados vem quase toda dos pontos com $C_D$ grande, em $\text{Re} < 10$. Errar 0,2 num ponto em que $C_D = 0{,}17$ não pesa nada no $R^2$, mas é um erro de mais de 100 % na força.

**Dentro da rede.** Por que a curva fica constante? Basta olhar a primeira camada:

In [ ]:
with torch.no_grad():
    z = modelo[0](X)                                      # entradas dos 32 neurônios da 1ª camada
saturados = (z.abs() > 3).double().mean(dim=1).numpy()    # fração com |z| > 3, onde tanh ≈ ±1
for faixa, msk in [("Re < 10", Re_med < 10), ("10 < Re < 10³", (Re_med > 10) & (Re_med < 1e3)), ("Re > 10³", Re_med > 1e3)]:
    print(f"{faixa:14s}: {saturados[msk].mean():.0%} dos neurônios da 1ª camada saturados")

### 6.3 Diagnóstico

Com $\text{Re}$ entrando cru, de $0{,}1$ a $10^6$, a primeira camada recebe $w_j\,\text{Re} + b_j$ com valores enormes para quase todos os dados. Para $\text{Re} > 10^3$, todos os neurônios estão saturados em $\pm 1$: para a rede, $\text{Re} = 10^4$ e $\text{Re} = 10^6$ são **a mesma entrada**, e nenhum ajuste dos pesos pode separá-los. Além disso, a perda em $C_D$ absoluto é dominada pelos pontos de $\text{Re}$ baixo. São os dois problemas de escala da seção 3.3. O código está correto; o problema de otimização que ele monta é **mal condicionado**.

### 6.4 A correção

Ajustar $\log_{10} C_D$ em função de $\log_{10}\text{Re}$, as duas padronizadas:

In [ ]:
torch.manual_seed(0)
px, py = Padronizador(np.log10(Re_med)), Padronizador(np.log10(Cd_med))
rede_log = mlp(largura=32, camadas=3)
r_log = treinar(rede_log, px(np.log10(Re_med)), py(np.log10(Cd_med)), epocas=5000, lr=1e-2)

def cd_rede_log(Re):
    with torch.no_grad():
        return 10 ** py.inverso(rede_log(px(np.log10(np.atleast_1d(Re))))).numpy().ravel()

erro_dados = np.abs(cd_rede_log(Re_med) / Cd_med - 1)
erro_curva = np.abs(cd_rede_log(Re_curva) / cd_correlacao(Re_curva) - 1)
print(f"erro relativo nos dados: mediana {np.median(erro_dados):.1%}, máximo {erro_dados.max():.1%}  (o ruído é de 3 %)")
print(f"erro relativo contra a curva verdadeira: mediana {np.median(erro_curva):.1%}, máximo {erro_curva.max():.1%}")
for Re_ in [0.1, 3e5]:
    print(f"Re = {Re_:g}: rede {cd_rede_log(Re_)[0]:.3f}, curva verdadeira {cd_correlacao(Re_):.3f}")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=Re_curva, y=cd_correlacao(Re_curva), mode="lines", name="curva verdadeira",
                         line=dict(color=CORES["exata"], dash="dot")))
fig.add_trace(go.Scatter(x=Re_med, y=Cd_med, mode="markers", name="medidas", marker=dict(color=CORES["exata"], size=6)))
fig.add_trace(go.Scatter(x=Re_curva, y=cd_rede(Re_curva), mode="lines", name="rede do agente (Re cru)",
                         line=dict(color=CORES["erro"], width=2)))
fig.add_trace(go.Scatter(x=Re_curva, y=cd_rede_log(Re_curva), mode="lines", name="rede em log, padronizada",
                         line=dict(color=CORES["aproximacao"], width=3)))
fig.update_xaxes(type="log", title="número de Reynolds", exponentformat="power")
fig.update_yaxes(type="log", title="C_D", range=[-1.2, 2.6])
fig.update_layout(width=800, height=440, legend=dict(x=0.55, y=0.98))
mostrar(fig)

A mesma arquitetura, com as variáveis certas, acompanha os três regimes e a crise do arrasto, com erros do tamanho do ruído das medidas. O que muda na prática? A força de arrasto numa bola de 22 cm chutada a 25 m/s no ar:

In [ ]:
Re_bola = 1.2 * 25 * 0.22 / 1.8e-5                       # ρ v d / μ do ar
area_bola = np.pi * 0.22**2 / 4
print(f"Re da bola = {Re_bola:.2e}")
for nome, cd in [("curva verdadeira", cd_correlacao), ("rede do agente", cd_rede), ("rede corrigida", cd_rede_log)]:
    C_D = float(np.ravel(cd(Re_bola))[0])
    print(f"{nome:17s}: C_D = {C_D:.3f};  arrasto = {C_D * 0.5 * 1.2 * 25**2 * area_bola:.1f} N")

Com a rede do agente, o simulador de trajetória aplicaria quase quatro vezes o arrasto real: a bola cairia muito antes. A rede corrigida erra menos que o ruído das medidas.

> **Lição.** Uma rede neural é um modelo de mínimos quadrados não linear, e herda todos os cuidados do capítulo 7, mais os de otimização do capítulo 5. Antes de treinar, **padronize** as variáveis e use logaritmos para grandezas que variam em ordens de grandeza. Depois de treinar, não confie na perda nem no $R^2$: olhe o gráfico na escala certa, calcule o **erro relativo**, confira **limites físicos** conhecidos e separe dados de **validação**.

## 7. Quando falha

### 7.1 Fora dos dados, a rede inventa

Treinamos cinco redes iguais, mudando só a semente aleatória dos pesos iniciais, e comparamos com a física fora da faixa medida: Stokes para $\text{Re} \to 0$.

In [ ]:
Re_largo = np.logspace(-3, 8, 300)
fig = go.Figure()
fig.add_trace(go.Scatter(x=Re_largo, y=24 / Re_largo, mode="lines", name="Stokes: 24/Re",
                         line=dict(color=CORES["referencia"], dash="dash")))
fig.add_trace(go.Scatter(x=Re_med, y=Cd_med, mode="markers", name="medidas", marker=dict(color=CORES["exata"], size=5)))
previsoes = []
for semente in range(5):
    torch.manual_seed(semente)
    rede_s = mlp(largura=32, camadas=3)
    treinar(rede_s, px(np.log10(Re_med)), py(np.log10(Cd_med)), epocas=3000, lr=1e-2)
    with torch.no_grad():
        prev = 10 ** py.inverso(rede_s(px(np.log10(Re_largo)))).numpy().ravel()
    previsoes.append(prev)
    fig.add_trace(go.Scatter(x=Re_largo, y=prev, mode="lines", name="redes (5 sementes)", showlegend=(semente == 0),
                             line=dict(color=CORES["aproximacao"], width=1.5)))
fig.add_vrect(x0=0.1, x1=1e6, fillcolor="rgba(0,0,0,0.04)", line_width=0)
fig.update_xaxes(type="log", title="número de Reynolds", exponentformat="power")
fig.update_yaxes(type="log", title="C_D", range=[-2, 4.5])
fig.update_layout(width=820, height=460, legend=dict(x=0.6, y=0.98), title="Faixa sombreada: onde há medidas")
mostrar(fig)
previsoes = np.array(previsoes)
for Re_ in [1e-3, 1.0, 1e8]:
    i = np.argmin(np.abs(Re_largo - Re_))
    print(f"Re = {Re_:g}: redes de {previsoes[:, i].min():.3g} a {previsoes[:, i].max():.3g};  Stokes/medidas: "
          f"{24 / Re_ if Re_ < 1 else cd_correlacao(Re_):.3g}  (referência)")

Dentro da faixa medida, as cinco redes concordam. Fora dela, cada uma segue numa direção, e nenhuma sabe da lei de Stokes: uma rede com $\tanh$ tende a uma **constante** longe dos dados, enquanto $C_D$ cresce como $24/\text{Re}$. Um polinômio extrapolaria mal também (capítulo 6), mas uma correlação com a forma física certa, como $24/\text{Re} + \ldots$, extrapola bem. A dispersão entre sementes é um indicador barato de incerteza: onde as redes discordam, o modelo não sabe.

### 7.2 Uma PINN que não aprende

Na aleta da seção 5.2, troque $mL = 1{,}5$ por $mL = 5$, uma aleta longa, em que a temperatura cai depressa perto da base:

In [ ]:
for mL in [1.5, 5.0]:
    theta_mL, hist_mL, _ = treina_pinn(mL, epocas=4000)
    print(f"mL = {mL}: perda final {hist_mL[-1]:.1e};  erro máximo {np.abs(theta_mL - exata_aleta(mL)).max():.1e}")

# o mesmo problema por diferenças finitas (capítulo 14): um sistema tridiagonal
N_df = 200
dxi = 1 / N_df
A_df = (np.diag(np.full(N_df, -2 - (5.0 * dxi) ** 2)) + np.diag(np.ones(N_df - 1), 1) + np.diag(np.ones(N_df - 1), -1))
A_df[-1, -2] = 2.0                                     # θ'(1) = 0 por um ponto fantasma
b_df = np.zeros(N_df); b_df[0] = -1.0                  # θ(0) = 1
theta_df = np.r_[1.0, np.linalg.solve(A_df, b_df)]
print(f"diferenças finitas, mL = 5: erro máximo {np.abs(theta_df - np.cosh(5 * (1 - np.linspace(0, 1, N_df + 1))) / np.cosh(5)).max():.1e}")

Com $mL = 5$, a PINN para numa solução errada, com erro da ordem da própria solução. O termo da equação, multiplicado por $(mL)^2 = 25$, domina a perda e empurra a rede para $\theta \approx 0$, que satisfaz a equação em quase todo lugar, sacrificando a condição de contorno. Há remédios (pesos para cada termo, mudança de escala de $\xi$, mais pontos perto da base), mas cada um é mais um parâmetro a ajustar. As diferenças finitas resolvem o mesmo problema em milissegundos, com erro controlado e previsível.

### 7.3 O que a rede não garante

- **Conservação e sinais.** Nada impede a rede de prever $C_D < 0$ ou uma eficiência acima de 1 fora dos dados. Restrições físicas precisam ser impostas (por exemplo, prever $\log C_D$ garante $C_D > 0$).
- **Reprodutibilidade.** Sem fixar sementes, dois treinos dão redes diferentes. Para um relatório, registre sementes, versões e o histórico da perda.
- **Erro de truncamento.** Uma PINN não tem uma "ordem" que diga quanto o erro cai ao refinar; a verificação precisa ser feita contra soluções conhecidas, como em qualquer método novo.

## Resumo

| Conceito | No ajuste de curvas (cap. 7) | Na rede neural |
| --- | --- | --- |
| Modelo | linear nos parâmetros (polinômio, base) | composição de camadas, não linear |
| Ajuste | equações normais, QR | descida pelo gradiente (Adam), gradiente por dif. automática |
| Condicionamento | escalar colunas, evitar Vandermonde | padronizar entradas e saídas; logaritmos |
| Sobreajuste | grau alto demais | parâmetros demais, treino longo demais; validação |
| Regularização | Tikhonov ($\lambda\|\mathbf{c}\|^2$) | decaimento de pesos, parada antecipada |
| Física | escolher a forma do modelo | PINN: a equação entra na perda |

## Exercícios

**Conceituais (sem computador)**

1. Conte os parâmetros de uma rede com 2 entradas, 2 camadas ocultas de 16 neurônios e 1 saída.
2. Mostre que, se as ativações fossem lineares ($\sigma(z) = z$), uma rede com qualquer número de camadas seria equivalente a um modelo linear.
3. Refaça a seção 4 com a ativação ReLU no lugar de $\tanh$. O que acontece com o gradiente se $z < 0$?
4. Por que minimizar o erro quadrático em $\log C_D$ equivale, aproximadamente, a minimizar o erro relativo em $C_D$?

**Verificação (com agente)**

5. Peça a um agente um ajuste de $C_D(\text{Re})$ por um polinômio em $\log\text{Re}$ e compare com a rede: número de parâmetros, erro relativo e comportamento fora dos dados.
6. Peça a um agente para treinar a rede da seção 5.1 com parada antecipada (*early stopping*). Verifique se a época escolhida coincide com o mínimo da validação e se o código usa os dados de validação **só** para escolher a época.
7. Peça a um agente uma PINN para a aleta com $mL = 5$ que funcione. Verifique contra a solução exata e conte quantas tentativas (pesos, escalas, pontos) foram necessárias.

**Projeto aberto**

8. **Problema inverso com PINN.** Uma aleta tem $\theta$ medida em 5 pontos, com ruído de 1 %. Faça $mL$ um parâmetro treinável da PINN, junto com os pesos, e estime $mL$ (e daí o coeficiente de convecção $h$). Compare com um ajuste direto de $\cosh(mL(1 - \xi))/\cosh(mL)$ por mínimos quadrados não lineares (capítulo 7). Qual é mais preciso? Qual generaliza para uma aleta de seção variável, sem solução exata?

## Leituras

- F. A. Morrison, *Data Correlation for Drag Coefficient for Sphere*, Michigan Technological University, 2013 (a correlação usada nos dados sintéticos).
- I. Goodfellow, Y. Bengio e A. Courville, *Deep Learning*, MIT Press, 2016, capítulos 6 a 8.
- M. Raissi, P. Perdikaris e G. E. Karniadakis, "Physics-informed neural networks", *Journal of Computational Physics*, 378, 2019.
- S. Wang, Y. Teng e P. Perdikaris, "Understanding and mitigating gradient flow pathologies in physics-informed neural networks", *SIAM Journal on Scientific Computing*, 43(5), 2021.